# PT1 Tiefpassfilter

Lineares Tiefpassfilter erster Ordnung (PT1-Tiefpassfilter)

$$
\begin{align}
y(k) &= y(k-1) + \frac{dT}{T}\left(u(k)-y(k-1)\right)\\
y(k) &= \left(1- \frac{dT}{T}\right)y(k-1) + \frac{dT}{T}u(k)\\
\end{align}
$$
mit Eingang $u(k)$ und Ausgang $y(k)$ zum Zeitpunkt $k$ und $T$ als Zeitkontante des Filters und $dT$ der Abtastzeit.

Inhalt:
- Realisierung des PT1-Tiefpassfilter als Python-Funktion
  - Programmierung mit for Schleife
  - Programmierung unter Verwendung von `scipy.signal.lfilter()`
- Sprungantwort
- Frequenz- und Phasengang


----
2026-06-07 ug V1.3

In [ ]:
import numpy as np 
import scipy.signal as signal  
from matplotlib import pyplot as plt
%matplotlib inline

### Definition des linearen Filters erster Ordnung
    
$$
\begin{align}
y(k) &= y(k-1) + \frac{dT}{T}\left(u(k)-y(k-1)\right)\\
y(k) &= \left(1- \frac{dT}{T}\right)y(k-1) + \frac{dT}{T}u(k)\\
\end{align}
$$
mit Eingang $u(k)$ und Ausgang $y(k)$ zum Zeitpunkt $k$ und $T$ als Zeitkontants des Filters und $dT$ der Abtastzeit.


In [ ]:
def pt1_filter(t,u,T,u0=None):
    """
    first order linear filter
    inputs:
        t  - time           numpy array
        u  - input signals  numpy array
        T  - time constant  numpy array
        u0 - intial state   scalar
    output:
        y 
    """

    y = np.zeros_like(u)

    # initial state
    if u0:
        y[0] = u0
    else:  
        y[0] = u[0]
    
    for k in range(1,u.size):
        y[k] = y[k-1] + (t[k]-t[k-1])/T*(u[k]-y[k-1])  

    return y  


### Alternative Berechnung des PT1-Filters mit der Funktion `scipy.signal.lfilter()`:

[`scipy.signal.lfilter()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.lfilter.html)

Herleiten der Filterkoeffizienten:
$$
\begin{align}
y(k)+\left(\frac{dT}{T} - 1 \right) y(k-1) &= \frac{dT}{T}u(k)\\
y(k)\left[1 + \left(\frac{dT}{T}-1\right) z^{-1}\right] &= \frac{dT}{T}u(k)\\
y(k)\left[a_0 + a_1 z^{-1}\right] &= b_0 u(k)
\end{align}
$$

mit 
$$
\begin{align}
    a_0 &= 1\\
    a_1 &= \frac{dT}{T}-1\\
    b_0 &= \frac{dT}{T}
\end{align}
$$

In [ ]:
def pt1_filter_lfilter(t,u,T,u0=None):
    """
    first order linear filter
    inputs:
        t  - time           numpy array
        u  - input signals  numpy array
        T  - time constant  numpy array
        u0 - intial state   scalar
    output:
        filtered signal
    """

    B = np.array([dT/T])
    A = np.array([1, dT/T-1.0])
    if u0 is None:
        return signal.lfilter(B,A,u)
    else:
        return signal.lfilter(B,A,u,zi=np.array([u0]))[0]

---
### Sprungantwort

In [ ]:
N = 200
dT = 0.01  
print(f"Abtastfrequenz: {1/dT} Hz")

# Zeitsignal
t = np.arange(0,N*dT,dT)

# Sprung auf 1 
t_start_step = 0.3

input_signal = np.zeros_like(t)
input_signal[t>t_start_step] = 1.0

#### Testsignal plotten

In [ ]:
fig, ax1 = plt.subplots(figsize=(16,8))

ax1.set_title('Testsignal Sinus)')
ax1.plot(t,input_signal,color='b',label='Eingang')

ax1.set_ylim((-0.1, 1.5))
ax1.grid()
ax1.set_xlabel('Zeit [s]')
ax1.set_ylabel('Eingangssignal ')
ax1.legend();

Zeitkontante $T$= 200ms

In [ ]:
T = 0.2

In [ ]:
output_signal = pt1_filter(t,input_signal,T)

In [ ]:
output_signal2 = pt1_filter_lfilter(t,input_signal,T)

In [ ]:
def plot_sprung():
    fig, ax1 = plt.subplots(figsize=(16,8))

    t_63 = t[output_signal>=0.63][0]-t_start_step
    print(f"t bis auf 63% der Amplitude {t_63:.3f} s")

    ax1.set_title("PT1 Filter")
    ax1.plot(t - t_start_step,input_signal,color='b',label='Eingang')
    ax1.plot(t - t_start_step,output_signal,color='r',label='Ausgang')
    ax1.plot(t - t_start_step,output_signal2,color='green',label='Ausgang2')

    ax1.plot(t - t_start_step, np.ones_like(t) * 0.63, color='m',ls='--',label=f'63% - {t_63:.3f} s') 

    ax1.set_ylim((-0.1, 1.2))
    ax1.set_xlim((-0.4, 2.0))
    ax1.grid()
    ax1.set_xlabel('Zeit [s]')
    ax1.set_ylabel('Ein-/Ausgangssignal ')
    ax1.legend();

In [ ]:
plot_sprung()

PT1 Filter Berechnungen stimmen überein.

### Filter-Berechnung mit Startwerten

In [ ]:
Initialwert = 0.5

output_signal = pt1_filter(t,input_signal,T,u0=Initialwert)
output_signal2 = pt1_filter_lfilter(t,input_signal,T,u0=Initialwert)
plot_sprung()

# Frequenz- und Phasengang

Grenzfrequenz $f_g$ eines linearen Filters erster Ordnung:
$$
    f_g = \frac{1}{2 \pi} \cdot \frac{1}{T}
$$
mit $T$ als Zeitkonstante.

Winkel einer komplexen Zahl bestimmen: [`np.angle()`](https://numpy.org/doc/stable/reference/generated/numpy.angle.html)

In [ ]:
fig, (ax1,ax2) = plt.subplots(nrows=2, figsize=(16,8), sharex=True)

dT = 0.01        # Abtastzeit in Sekunden
T  = 0.2         # Zeitkonstante in Sekunden

# ---------------------
fg = 1/T/2/np.pi         # Grenzfrequenz
print(f"Grenzfrequenz {fg:.3f} Hz")

fs = 1.0/dT         # sampling frequency (assumption: constant sampling interval)
f_nyq = fs/2.0      # Nyquist frequency (= 1/2 sampling frequency)

# Filterkoeffizienten - PT1 Filter
B = np.array([dT/T])
A = np.array([1, dT/T-1.0])

# Berechnung des Frequenzgangs
w,h = signal.freqz(B, A)

# w ist eine normierte Frequenz und geht von 0 bis pi; pi entspricht der Nyquistfrequenz
f = w/np.pi*f_nyq

# Amplitudengang
ax1.semilogx(f, 20 * np.log10(abs(h)),label=f'PT1 Filter')
ax1.axhline(-3.0, color='black',ls=':',label='3dB')
ax1.axvline(fg, color='red', label=f'Grenzfrequenz {fg:.3f} Hz')
ax1.set_ylim(-50,10)

ax1.set_title('PT1 Filter Amplituden und Phasengang')
ax1.set_ylabel('Amplitude [dB]')
ax1.legend()

#plt.margins(0, 0.1)
ax1.grid(which='both', axis='both')

#  Phasengang
ax2.semilogx(f, np.angle(h,deg=True),label=f'PT1 Filter')
ax2.axhline(-45, color='black',ls=':',label='45°')
ax2.axvline(fg, color='red', label=f'Grenzfrequenz {fg:.3f} Hz') 
ax2.grid(which='both', axis='both')
ax2.set_xlabel('Frequency [Hz]')
ax2.set_ylabel('Phase [°]')
ax2.legend();
